# Deep Reinforcement Learning for Trading

**Paper:** Zhang, Zohren, and Roberts (Oxford, 2019)  
**Link:** https://arxiv.org/pdf/1911.10107

This notebook implements DRL trading strategies with GPU support for cloud deployment.

## Contents
1. Setup & Data Loading
2. Baseline Strategies (Long, Sign, MACD)
3. DRL Agents (DQN, PPO, A2C)
4. Evaluation & Comparison
5. Results Export

## 1. Setup & Configuration

In [ ]:
# Install required packages (run once on cloud)
!pip install -q yfinance stable-baselines3 gymnasium pandas numpy matplotlib tqdm

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    DEVICE = 'cuda'
else:
    print("Running on CPU")
    DEVICE = 'cpu'

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm

# DRL libraries
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.callbacks import EvalCallback

# Configuration
TRAIN_START = '2011-01-01'
TRAIN_END = '2017-06-30'
TEST_START = '2017-07-01'
TEST_END = '2019-12-31'

TRANSACTION_COST = 0.001  # 10 bps per trade (per paper)
LOOKBACK = 50  # Days of history for state

# Create output directories
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data/futures', exist_ok=True)

print("✅ Setup complete!")

## 2. Data Loading

In [ ]:
# Paper's 50 futures contracts mapping (Pinnacle -> Yahoo)
PAPER_CONTRACTS = {
    # Commodities (25)
    'CC=F': 'COCOA', 'OJ=F': 'ORANGE JUICE', 'KC=F': 'COFFEE', 'LBS=F': 'LUMBER',
    'ZR=F': 'ROUGH RICE', 'SB=F': 'SUGAR', 'PA=F': 'PALLADIUM', 'ZC=F': 'CORN',
    'GF=F': 'FEEDER CATTLE', 'GC=F': 'GOLD', 'HO=F': 'HEATING OIL', 'SI=F': 'SILVER',
    'HG=F': 'COPPER', 'ZL=F': 'SOYBEAN OIL', 'NG=F': 'NATURAL GAS', 'ZO=F': 'OATS',
    'PL=F': 'PLATINUM', 'LE=F': 'LIVE CATTLE', 'CL=F': 'CRUDE OIL', 'ZW=F': 'WHEAT',
    'HE=F': 'LEAN HOGS',
    # Equity Indexes (6 available)
    'NQ=F': 'NASDAQ', 'RTY=F': 'RUSSELL 2000', 'ES=F': 'S&P 500', 'YM=F': 'DOW JONES',
    'NKD=F': 'NIKKEI',
    # Fixed Income (3 available)
    'ZF=F': '5Y TREASURY', 'ZN=F': '10Y TREASURY', 'ZB=F': '30Y TREASURY',
    # Forex (9 available)
    '6A=F': 'AUD', '6B=F': 'GBP', '6C=F': 'CAD', 'DX=F': 'DOLLAR INDEX',
    '6E=F': 'EUR', '6J=F': 'JPY', '6M=F': 'MXN', '6S=F': 'CHF',
}

def download_futures_data(tickers, start='2011-01-01', end='2019-12-31'):
    """Download futures data from Yahoo Finance."""
    import yfinance as yf
    
    data = {}
    for ticker in tqdm(tickers, desc="Downloading"):
        try:
            df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if not df.empty:
                df['Returns'] = df['Close'].pct_change()
                data[ticker] = df
        except Exception as e:
            print(f"Failed to download {ticker}: {e}")
    return data

# Check if data already exists
data_files = [f for f in os.listdir('data/futures') if f.endswith('.csv')]

if len(data_files) >= 30:
    print(f"✅ Found {len(data_files)} existing data files")
    # Load existing data
    futures_data = {}
    for f in data_files:
        ticker = f.replace('.csv', '')
        df = pd.read_csv(f'data/futures/{f}', skiprows=3, names=['Date', 'Close', 'High', 'Low', 'Open', 'Volume'])
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.set_index('Date').dropna()
        df['Returns'] = df['Close'].pct_change()
        futures_data[ticker] = df
else:
    print("📥 Downloading futures data...")
    futures_data = download_futures_data(PAPER_CONTRACTS.keys())
    # Save data
    for ticker, df in futures_data.items():
        df.to_csv(f'data/futures/{ticker}.csv')

print(f"✅ Loaded {len(futures_data)} futures contracts")

In [ ]:
# Split data into train/test
def get_train_test_split(df):
    train = df[(df.index >= TRAIN_START) & (df.index < TRAIN_END)].copy()
    test = df[(df.index >= TEST_START) & (df.index <= TEST_END)].copy()
    return train, test

# Select pilot contracts for testing
PILOT_CONTRACTS = ['ES=F', 'CL=F', 'GC=F', 'ZN=F', '6E=F']

# Verify data availability
for ticker in PILOT_CONTRACTS:
    if ticker in futures_data:
        train, test = get_train_test_split(futures_data[ticker])
        print(f"{ticker}: Train={len(train)} days, Test={len(test)} days")
    else:
        print(f"{ticker}: ❌ Not available")

## 3. Trading Environment (Gymnasium)

In [ ]:
class FuturesTradingEnv(gym.Env):
    """
    Custom trading environment for futures following paper's setup.
    
    State: Normalized returns over lookback period
    Action: Discrete (0=short, 1=neutral, 2=long) or Continuous (portfolio weight)
    Reward: Return - transaction cost
    """
    
    metadata = {'render_modes': ['human']}
    
    def __init__(self, returns, prices=None, lookback=50, transaction_cost=0.001,
                 mode='discrete'):
        super().__init__()
        
        self.returns = returns.values if hasattr(returns, 'values') else returns
        self.prices = prices.values if prices is not None and hasattr(prices, 'values') else prices
        self.lookback = lookback
        self.transaction_cost = transaction_cost
        self.mode = mode
        
        # Define action and observation space
        if mode == 'discrete':
            # 0=short, 1=neutral, 2=long
            self.action_space = spaces.Discrete(3)
        else:
            # Continuous: -1 to 1 (short to long)
            self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)
        
        # Observation: lookback days of normalized returns
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(lookback,), dtype=np.float32
        )
        
        self.reset()
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = self.lookback
        self.position = 0
        self.done = False
        self.total_reward = 0
        self.positions_history = []
        self.returns_history = []
        
        return self._get_obs(), {}
    
    def _get_obs(self):
        """Get normalized returns as observation."""
        obs = self.returns[self.t - self.lookback:self.t]
        # Normalize
        mean = np.mean(obs)
        std = np.std(obs) + 1e-8
        obs = (obs - mean) / std
        return obs.astype(np.float32)
    
    def step(self, action):
        # Convert action to position
        if self.mode == 'discrete':
            new_position = action - 1  # -1, 0, 1
        else:
            new_position = float(action[0])
        
        # Calculate transaction cost
        trade_cost = abs(new_position - self.position) * self.transaction_cost
        
        # Get return
        ret = self.returns[self.t]
        
        # Calculate reward
        reward = new_position * ret - trade_cost
        
        # Update state
        self.position = new_position
        self.t += 1
        self.total_reward += reward
        self.positions_history.append(new_position)
        self.returns_history.append(reward)
        
        # Check if done
        if self.t >= len(self.returns) - 1:
            self.done = True
        
        # Additional info
        info = {
            'position': self.position,
            'return': ret,
            'reward': reward,
            'total_reward': self.total_reward
        }
        
        return self._get_obs(), reward, self.done, False, info

print("✅ Trading environment defined!")

## 4. Baseline Strategies

In [ ]:
def strategy_long(returns):
    """Buy and hold."""
    return np.ones(len(returns))

def strategy_sign(returns, lookback=50):
    """Momentum: sign of past returns."""
    signals = np.zeros(len(returns))
    for i in range(lookback, len(returns)):
        past_return = np.sum(returns[i-lookback:i])
        signals[i] = np.sign(past_return)
    return signals

def strategy_macd(prices, fast=12, slow=26, signal=9):
    """MACD crossover."""
    ema_fast = prices.ewm(span=fast).mean()
    ema_slow = prices.ewm(span=slow).mean()
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal).mean()
    signals = np.where(macd > signal_line, 1, -1)
    return signals

def calculate_metrics(returns, positions):
    """Calculate performance metrics per paper's Table 2."""
    portfolio_returns = returns * positions
    portfolio_returns = np.roll(portfolio_returns, -1)[:-1]
    
    annual_return = np.mean(portfolio_returns) * 252
    annual_std = np.std(portfolio_returns) * np.sqrt(252)
    sharpe = annual_return / annual_std if annual_std > 0 else 0
    
    cumulative = np.cumprod(1 + portfolio_returns)
    running_max = np.maximum.accumulate(cumulative)
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = np.min(drawdown)
    
    calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    downside_returns = portfolio_returns[portfolio_returns < 0]
    downside_std = np.std(downside_returns) * np.sqrt(252) if len(downside_returns) > 0 else 1
    sortino = annual_return / downside_std if downside_std > 0 else 0
    
    return {
        'E(R)': annual_return,
        'Std(R)': annual_std,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'MDD': max_drawdown,
        'Calmar': calmar
    }

print("✅ Baseline strategies defined!")

## 5. DRL Agent Training

In [ ]:
def train_drl_agents(ticker, futures_data, total_timesteps=50000):
    """
    Train DQN, PPO, and A2C agents for a single futures contract.
    """
    df = futures_data[ticker]
    train, test = get_train_test_split(df)
    
    train_returns = train['Returns'].dropna()
    test_returns = test['Returns'].dropna()
    test_prices = test['Close']
    
    # Create environments
    train_env = FuturesTradingEnv(train_returns, lookback=LOOKBACK, 
                                   transaction_cost=TRANSACTION_COST, mode='discrete')
    test_env = FuturesTradingEnv(test_returns, lookback=LOOKBACK,
                                  transaction_cost=TRANSACTION_COST, mode='discrete')
    
    results = {'ticker': ticker}
    
    # ========== Baselines ==========
    print(f"\n📊 {ticker} - Baseline Strategies")
    
    # Long
    positions = strategy_long(test_returns.values)
    results['Long'] = calculate_metrics(test_returns.values, positions)
    print(f"  Long: Sharpe={results['Long']['Sharpe']:.3f}")
    
    # Sign
    positions = strategy_sign(test_returns.values, lookback=50)
    results['Sign'] = calculate_metrics(test_returns.values, positions)
    print(f"  Sign: Sharpe={results['Sign']['Sharpe']:.3f}")
    
    # MACD
    positions = strategy_macd(test_prices)
    results['MACD'] = calculate_metrics(test_returns.values, positions)
    print(f"  MACD: Sharpe={results['MACD']['Sharpe']:.3f}")
    
    # ========== DRL Agents ==========
    agents = {
        'DQN': DQN('MlpPolicy', train_env, verbose=0, device=DEVICE,
                   learning_rate=1e-3, buffer_size=10000, learning_starts=1000),
        'PPO': PPO('MlpPolicy', train_env, verbose=0, device=DEVICE,
                   learning_rate=3e-4, n_steps=2048),
        'A2C': A2C('MlpPolicy', train_env, verbose=0, device=DEVICE,
                   learning_rate=7e-4, n_steps=5)
    }
    
    for name, agent in agents.items():
        print(f"\n🤖 Training {name}...")
        
        # Train
        agent.learn(total_timesteps=total_timesteps)
        
        # Test
        obs, _ = test_env.reset()
        done = False
        positions = []
        
        while not done:
            action, _ = agent.predict(obs, deterministic=True)
            obs, _, done, _, _ = test_env.step(action)
            positions.append(action - 1)  # Convert to -1, 0, 1
        
        positions = np.array(positions)
        results[name] = calculate_metrics(test_returns.values[LOOKBACK:LOOKBACK+len(positions)], positions)
        print(f"  {name}: Sharpe={results[name]['Sharpe']:.3f}")
        
        # Save model
        agent.save(f'models/{ticker.replace("=F", "")}_{name}')
    
    return results

print("✅ Training function defined!")

In [ ]:
# Train on all pilot contracts
all_results = {}

for ticker in PILOT_CONTRACTS:
    if ticker in futures_data:
        print(f"\n{'='*60}")
        print(f"📊 Training on {ticker} ({PAPER_CONTRACTS.get(ticker, 'Unknown')})")
        print(f"{'='*60}")
        
        results = train_drl_agents(ticker, futures_data, total_timesteps=50000)
        all_results[ticker] = results

print("\n✅ Training complete!")

## 6. Results Summary

In [ ]:
# Create results summary table
print("\n" + "="*80)
print("📊 PERFORMANCE SUMMARY (Test Period: 2017-07-01 to 2019-12-31)")
print("="*80)

summary_data = []
for ticker, results in all_results.items():
    for strategy in ['Long', 'Sign', 'MACD', 'DQN', 'PPO', 'A2C']:
        if strategy in results:
            metrics = results[strategy]
            summary_data.append({
                'Ticker': ticker,
                'Strategy': strategy,
                'Sharpe': f"{metrics['Sharpe']:.3f}",
                'Sortino': f"{metrics['Sortino']:.3f}",
                'MDD': f"{metrics['MDD']:.2%}",
                'Calmar': f"{metrics['Calmar']:.3f}"
            })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save results
summary_df.to_csv('results/performance_summary.csv', index=False)

with open('results/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"\n💾 Results saved to results/")

In [ ]:
# Visualize Sharpe ratios
fig, ax = plt.subplots(figsize=(12, 6))

strategies = ['Long', 'Sign', 'MACD', 'DQN', 'PPO', 'A2C']
x = np.arange(len(PILOT_CONTRACTS))
width = 0.12

for i, strategy in enumerate(strategies):
    sharpes = []
    for ticker in PILOT_CONTRACTS:
        if ticker in all_results and strategy in all_results[ticker]:
            sharpes.append(all_results[ticker][strategy]['Sharpe'])
        else:
            sharpes.append(0)
    ax.bar(x + i * width, sharpes, width, label=strategy)

ax.set_xlabel('Futures Contract')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio by Contract and Strategy')
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels([t.replace('=F', '') for t in PILOT_CONTRACTS])
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/sharpe_comparison.png', dpi=150)
plt.show()

print("\n📊 Chart saved to results/sharpe_comparison.png")

## 7. Extended Training on All Contracts (Optional)

In [ ]:
# Train on all available contracts (longer training time)
EXTENDED_TRAINING = False  # Set to True for full training

if EXTENDED_TRAINING:
    print("\n🚀 Extended Training on All Contracts")
    print("="*60)
    
    for ticker in futures_data.keys():
        if ticker not in all_results:
            print(f"\n📊 Training on {ticker}...")
            try:
                results = train_drl_agents(ticker, futures_data, total_timesteps=100000)
                all_results[ticker] = results
            except Exception as e:
                print(f"❌ Failed: {e}")
    
    # Save updated results
    with open('results/all_results_extended.json', 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print("\n✅ Extended training complete!")

## 8. Export for Local Use

In [ ]:
# Create a zip file of all results for download
import shutil

shutil.make_archive('drl_trading_results', 'zip', 'results')
shutil.make_archive('drl_trading_models', 'zip', 'models')

print("✅ Results exported!")
print("  - drl_trading_results.zip")
print("  - drl_trading_models.zip")